# Primero recopilamos


In [1]:
from pathlib import Path
import pandas as pd


In [5]:

carpeta = Path(".")

archivo_catalogo = carpeta / "formulario_limpio_1.xlsx"
sheet_catalogo = "Catálogo VRAI"

archivos_output = list(carpeta.glob("output_*.xlsx")) + list(carpeta.glob("output_single_*.xlsx"))
archivos_output = [f for f in archivos_output if f.name != archivo_catalogo.name]

# Catálogo VRAI: mapeo por NRC
df_catalogo = pd.read_excel(archivo_catalogo, sheet_name=sheet_catalogo, dtype=str)

columnas_catalogo = [
    "NRC",
    "Sigla",
    "Horario Cátedra/Clase",
    "Horario Ayudantía",
    "Horario Laboratorio",
    "Horario Taller",
    "Horario Terreno",
]

mapa_catalogo = (
    df_catalogo[columnas_catalogo]
    .dropna(subset=["NRC"])
    .assign(NRC=lambda x: x["NRC"].astype(str).str.strip())
    .drop_duplicates(subset=["NRC"], keep="first")
)

# Archivos output: filas a recopilar
columnas_output = [
    "RUT UC",
    "Nombre",
    "Email",
    "NRC",
    "Unidad Académica",
]

dfs = []

for archivo in archivos_output:
    df = pd.read_excel(archivo, dtype=str, sheet_name="Con éxito")

    df = df[columnas_output].copy()
    df["NRC"] = df["NRC"].astype(str).str.strip()

    dfs.append(df)

df_final = pd.concat(dfs, ignore_index=True)

# Cruce por NRC
df_final = df_final.merge(mapa_catalogo, on="NRC", how="left")

# Orden final
df_final = df_final[
    [
        "RUT UC",
        "Nombre",
        "Email",
        "NRC",
        "Unidad Académica",
        "Sigla",
        "Horario Cátedra/Clase",
        "Horario Ayudantía",
        "Horario Laboratorio",
        "Horario Taller",
        "Horario Terreno",
    ]
]

In [6]:
df_final

,RUT UC,Nombre,Email,NRC,Unidad Académica,Sigla,Horario Cátedra/Clase,Horario Ayudantía,Horario Laboratorio,Horario Taller,Horario Terreno
0,118606K,Elaine Victoria Garcia,elaine.garcia@tufts.edu,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
1,1185799,Mathilde Marie Blandine Lagarde,mathilde.lagrd@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
2,118539K,Vadim MAURER-DECOUT,vadim.maurerdecout@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
3,1186299,Ava Louise Giere,a.giere@wustl.edu,10218,Letras,LET1005,CLAS = L:1100 a 1210; W:1100 a 1210;,NaN,NaN,TAL = V:0940 a 1050;,NaN
4,1184148,Clara Fornés Roig,fornesroigc@gmail.com,10261,Estetica,EST210A,CLAS = M:1450 a 1720;,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
595,1184598,Julian Staeva-Vieira,julian.staeva_vieira@tufts.edu,29824,Agronomia,AGC209,CLAS = L:1100 a 1330;,NaN,NaN,NaN,NaN
596,1186892,Joana Haristoy,joana.haristoy@opendeusto.es,30100,Escuela de Medicina,MED855,CLAS = J:1100 a 1330;,NaN,NaN,NaN,NaN
597,1184520,Leslie Santiago Andres,lc23-0707@lclark.edu,30374,Ciencia Politica,ICP0720,CLAS = L:1220 a 1330; W:1220 a 1330;,NaN,NaN,NaN,NaN
598,1185748,Ximena Herrera Monreal,A01705214@itesm.mx,33476,Derecho,DNO158,CLAS = V:0820 a 1050;,NaN,NaN,NaN,NaN


In [7]:

# Guardar resultado
df_final.to_excel("recopilado_outputs.xlsx", index=False)

In [85]:
df_final = pd.read_excel("recopilado_outputs.xlsx")

# Ahora chequeamos

In [86]:
modulos = {
    1: "08:20",
    2: "09:40",
    3: "11:00",
    4: "12:20",
    5: "13:30",
    6: "14:50",
    7: "17:30",
    8: "18:50",
    9: "20:10"
}

In [87]:
archivo_eval = "Material Docente 24-02-2026_FACEA.xlsx"

df_eval = pd.read_excel(
    archivo_eval,
    skiprows=11,
    header=[0, 1]
)


In [88]:
df_eval

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
    Unnamed: 0_level_1              Sigla               Sec.   
0                  NaN            EAA1110                  1   
1                  NaN            EAA1110                  2   
2                  NaN            EAA1110                  3   
3                  NaN            EAA1110                  4   
4                  NaN            EAA1110                  5   
..                 ...                ...                ...   
382                NaN            EAG190A                  1   
383                NaN            EAG165E                  1   
384                NaN            EAG160E                  1   
385                NaN                NaN                NaN   
386                NaN                NaN                NaN   

    Unnamed: 3_level_0                                 Unnamed: 4_level_0  \
                  Créd                                   Nombre del curso   
0                   10                      COMPORTAMIENTO ORGANIZACIONAL   
1                   10                      COMPORTAMIENTO ORGANIZACIONAL   
2                   10                      COMPORTAMIENTO ORGANIZACIONAL   
3                   10                      COMPORTAMIENTO ORGANIZACIONAL   
4                   10                      COMPORTAMIENTO ORGANIZACIONAL   
..                 ...                                                ...   
382                 10  EXPLORANDO EL MUNDO DEL TRABAJO PARA LA TOMA D...   
383                 10  POLÍTICAS PÚBLICAS: DERRIBANDO MITOS DE LA POB...   
384                 10                          ¿CÓMO TOMAMOS DECISIONES?   
385                NaN                                                NaN   
386                NaN                                                NaN   

          Unnamed: 5_level_0                    Unnamed: 6_level_0  Clases  \
                        Prof                            Requisitos Horario   
0                   G. MUÑOZ                              ADMISIÓN   M-J:2   
1              S. VALENZUELA                              ADMISIÓN   M-J:4   
2              M.E. VIGNEAUX                              ADMISIÓN   M-J:4   
3                  E. KAUSEL                              ADMISIÓN   L-W:3   
4              S. VALENZUELA                              ADMISIÓN   M-J:5   
..                       ...                                   ...     ...   
382                  K. LUND                     CIENCIAS SOCIALES   M-J:5   
383  G. UGARTE / F. BASTIDAS                     CIENCIAS SOCIALES   L-W:2   
384                 C. MUÑOZ                     CIENCIAS SOCIALES   M-J:6   
385                      NaN                                   NaN     NaN   
386                      NaN  CURSOS DICTADOS POR OTRAS FACULTADES     NaN   

    Ayudantía                        Pruebas               \
      Horario Horario.1           Día/Módulo Día/Módulo.1   
0         V:6       NaN  29/abr:7 y 29/may:7          NaN   
1         V:6       NaN  29/abr:7 y 29/may:7          NaN   
2         V:6       NaN  29/abr:7 y 29/may:7          NaN   
3         V:6       NaN  29/abr:7 y 29/may:7          NaN   
4         V:6       NaN  29/abr:7 y 29/may:7          NaN   
..        ...       ...                  ...          ...   
382       V:5       NaN                   PF          NaN   
383       V:2       NaN             24/abr:2          NaN   
384         -       NaN   9/abr:6 y 14/may:6          NaN   
385       NaN       NaN                  NaN          NaN   
386       NaN       NaN                  NaN          NaN   

                  Examen                        Prueba Recuperativa            
                     Día           Hora Hora.1                  Día      Hora  
0    2024-07-03 00:00:00   8:20 - 10:20    NaN  2024-07-03 00:00:00  17:30:00  
1    2024-07-03 00:00:00   8:20 - 10:20    NaN  2024-07-03 00:00:00  17:30:00  
2    2024-07-03 00:00:00   8:20 - 10:20 

In [89]:

df_eval = df_eval[[("Unnamed: 1_level_0", "Sigla"),
                   ("Pruebas", "Día/Módulo"),
                   ("Examen", "Día"),
                   ("Examen", "Hora")]]

df_eval.columns = ["Sigla", "Pruebas", "Examen Dia", "Examen Hora"]
df_eval = df_eval.dropna(subset=["Sigla"]).reset_index(drop=True)
df_eval = df_eval.drop_duplicates(subset="Sigla").reset_index(drop=True)

In [90]:
df_eval

,Sigla,Pruebas,Examen Dia,Examen Hora
0,EAA1110,29/abr:7 y 29/may:7,2024-07-03 00:00:00,8:20 - 10:20
1,EAA1210,13/abr:7 y 26/may:7,2025-07-09 00:00:00,8:20 - 10:20
2,EAA1220,13/abr:7 y 26/may:7,2025-07-10 00:00:00,8:20 - 10:20
3,EAA2230,13/abr:7 y 26/may:7,2024-06-30 00:00:00,13:30 - 15:30
4,EAA220B,14/abr:7 y 1/jun:7,2024-07-09 00:00:00,13:30 - 15:30
...,...,...,...,...
169,CURSOS DE FORMACION GENERAL DICTADOS POR LA FA...,NaN,NaN,NaN
170,EAG170A,17/abr:2,2025-07-01 00:00:00,13:30 - 15:30
171,EAG190A,PF,PF,PF
172,EAG165E,24/abr:2,2025-07-01 00:00:00,13:30 - 15:30


In [91]:
siglas_relevantes = set(df_final["Sigla"].unique())

df_eval = df_eval[df_eval["Sigla"].isin(siglas_relevantes)].reset_index(drop=True)

In [92]:
df_eval

,Sigla,Pruebas,Examen Dia,Examen Hora
0,EAA1220,13/abr:7 y 26/may:7,2025-07-10 00:00:00,8:20 - 10:20
1,EAA2220,10/abr:7 y 15/may:7,2024-07-02 00:00:00,8:20 - 10:20
2,EAA2310,29/abr:7 y 29/may:7,2024-07-04 00:00:00,8:20 - 10:20
3,EAA2410,13/abr:7 y 26/may:7,2024-07-02 00:00:00,8:20 - 10:20
4,EAA2110,30/abr:7 y 11/jun:7,2024-07-03 00:00:00,8:20 - 10:20
5,EAA2420,23/abr:7 y 10/jun:7,2024-07-06 00:00:00,13:30 - 15:30
6,EAA1510,27/abr:7 y 5/jun:7,2024-07-08 00:00:00,13:30 - 15:30
7,EAA2081,PF,PF,PF
8,EAA2091,8/abr:7 y 2/jun:7,2025-06-30 00:00:00,8:20 - 10:20
9,EAA209L,PF,PF,PF


In [93]:

# Año a usar para las pruebas
anio_pruebas = 2026

meses = {
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
    "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12
}

dias_semana = {
    0: "L",
    1: "M",
    2: "W",
    3: "J",
    4: "V",
    5: "S",
    6: "D"
}

import re

def extraer_fechas_prueba(texto):
    
    if pd.isna(texto):
        return []

    texto = str(texto).strip()
    if texto == "PF":
        return []

    fechas = []

    # busca patrones tipo 13/abr:7, 8/abr:5-6, 8/abr:5; 13/abr:5, etc.
    coincidencias = re.findall(r"(\d{1,2})/([a-zA-Z]+)\s*:\s*([\d\-]+)", texto)

    for dia_txt, mes_txt, modulo_txt in coincidencias:
        dia = int(dia_txt)
        mes = meses[mes_txt.strip().lower()]
        fecha = pd.Timestamp(year=anio_pruebas, month=mes, day=dia)

        if "-" in modulo_txt:
            modulo_inicio = int(modulo_txt.split("-")[0])
        else:
            modulo_inicio = int(modulo_txt)

        fechas.append({
            "fecha": fecha,
            "dia_semana": dias_semana[fecha.weekday()],
            "modulo": modulo_inicio
        })

    return fechas


In [94]:


df_eval["Pruebas parseadas"] = df_eval["Pruebas"].apply(extraer_fechas_prueba)



In [95]:
df_eval["Pruebas parseadas"][20]

[{'fecha': Timestamp('2026-04-08 00:00:00'), 'dia_semana': 'W', 'modulo': 5},
 {'fecha': Timestamp('2026-04-13 00:00:00'), 'dia_semana': 'L', 'modulo': 5},
 {'fecha': Timestamp('2026-06-01 00:00:00'), 'dia_semana': 'L', 'modulo': 5}]

In [96]:
import re
from datetime import datetime, timedelta

columnas_horario = [
    "Horario Cátedra/Clase",
    "Horario Ayudantía",
    "Horario Laboratorio",
    "Horario Taller",
    "Horario Terreno",
]

def hhmm_a_min(hhmm):
    hhmm = str(hhmm).replace(":", "").strip()
    return int(hhmm[:2]) * 60 + int(hhmm[2:])

def bloques_desde_horario(texto):
    if pd.isna(texto):
        return []
    
    texto = str(texto).strip()
    if texto == "":
        return []

    coincidencias = re.findall(r'([LMWJV]):\s*(\d{3,4})\s*a\s*(\d{3,4})', texto)
    
    bloques = []
    for dia, inicio, fin in coincidencias:
        bloques.append({
            "dia_semana": dia,
            "inicio_min": hhmm_a_min(inicio),
            "fin_min": hhmm_a_min(fin)
        })
    return bloques


In [97]:

# bloques ocupados por fila/curso
df_final["bloques_ocupados"] = df_final[columnas_horario].apply(
    lambda fila: sum((bloques_desde_horario(fila[col]) for col in columnas_horario), []),
    axis=1
)


In [98]:
df_final

,RUT UC,Nombre,Email,NRC,Unidad Académica,Sigla,Horario Cátedra/Clase,Horario Ayudantía,Horario Laboratorio,Horario Taller,Horario Terreno,bloques_ocupados
0,118606K,Elaine Victoria Garcia,elaine.garcia@tufts.edu,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'M', 'inicio_min': 740, 'fin_m..."
1,1185799,Mathilde Marie Blandine Lagarde,mathilde.lagrd@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'M', 'inicio_min': 740, 'fin_m..."
2,118539K,Vadim MAURER-DECOUT,vadim.maurerdecout@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'M', 'inicio_min': 740, 'fin_m..."
3,1186299,Ava Louise Giere,a.giere@wustl.edu,10218,Letras,LET1005,CLAS = L:1100 a 1210; W:1100 a 1210;,NaN,NaN,TAL = V:0940 a 1050;,NaN,"[{'dia_semana': 'L', 'inicio_min': 660, 'fin_m..."
4,1184148,Clara Fornés Roig,fornesroigc@gmail.com,10261,Estetica,EST210A,CLAS = M:1450 a 1720;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'M', 'inicio_min': 890, 'fin_m..."
...,...,...,...,...,...,...,...,...,...,...,...,...
595,1184598,Julian Staeva-Vieira,julian.staeva_vieira@tufts.edu,29824,Agronomia,AGC209,CLAS = L:1100 a 1330;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'L', 'inicio_min': 660, 'fin_m..."
596,1186892,Joana Haristoy,joana.haristoy@opendeusto.es,30100,Escuela de Medicina,MED855,CLAS = J:1100 a 1330;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'J', 'inicio_min': 660, 'fin_m..."
597,1184520,Leslie Santiago Andres,lc23-0707@lclark.edu,30374,Ciencia Politica,ICP0720,CLAS = L:1220 a 1330; W:1220 a 1330;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'L', 'inicio_min': 740, 'fin_m..."
598,1185748,Ximena Herrera Monreal,A01705214@itesm.mx,33476,Derecho,DNO158,CLAS = V:0820 a 1050;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'V', 'inicio_min': 500, 'fin_m..."


In [99]:
df_final["bloques_ocupados"][0]

[{'dia_semana': 'M', 'inicio_min': 740, 'fin_min': 1040}]

In [100]:
df_final[df_final["Nombre"] == "Paul Eisenblätter"]

,RUT UC,Nombre,Email,NRC,Unidad Académica,Sigla,Horario Cátedra/Clase,Horario Ayudantía,Horario Laboratorio,Horario Taller,Horario Terreno,bloques_ocupados
210,1162969,Paul Eisenblätter,eisenblaetter.paul@gmail.com,27961,Economia y Administracion,EAA3501,CLAS = J:1750 a 1920;,AYU = V:1220 a 1330;,NaN,NaN,NaN,"[{'dia_semana': 'J', 'inicio_min': 1070, 'fin_..."
274,1162969,Paul Eisenblätter,eisenblaetter.paul@gmail.com,29585,Economia y Administracion,EAA3401,CLAS = L:1220 a 1330; W:1220 a 1330;,AYU = V:0820 a 0930;,NaN,NaN,NaN,"[{'dia_semana': 'L', 'inicio_min': 740, 'fin_m..."


In [101]:

# mapa de evaluaciones por sigla
mapa_pruebas = df_eval.set_index("Sigla")["Pruebas parseadas"].to_dict()


In [102]:
mapa_pruebas["EAA3401"]

[{'fecha': Timestamp('2026-05-28 00:00:00'), 'dia_semana': 'J', 'modulo': 7}]

In [103]:

def bloque_evaluacion(prueba):
    inicio = datetime.strptime(modulos[prueba["modulo"]], "%H:%M")
    fin = inicio + timedelta(hours=2)
    return {
        "dia_semana": prueba["dia_semana"],
        "inicio_min": inicio.hour * 60 + inicio.minute,
        "fin_min": fin.hour * 60 + fin.minute,
        "fecha": prueba["fecha"]
    }

def hay_tope(b1, b2):
    return (
        b1["dia_semana"] == b2["dia_semana"]
        and b1["inicio_min"] < b2["fin_min"]
        and b2["inicio_min"] < b1["fin_min"]
    )


In [105]:

topes = []

for rut, grupo in df_final.groupby("RUT UC"):
    cursos = grupo.to_dict("records")
    
    for curso_eval in cursos:
        sigla_eval = curso_eval["Sigla"]
        pruebas = mapa_pruebas.get(sigla_eval, [])
        
        if not pruebas:
            continue
        
        for prueba in pruebas:
            bloque_prueba = bloque_evaluacion(prueba)
            
            for otro_curso in cursos:
                if otro_curso["Sigla"] == sigla_eval:
                    continue
                
                for bloque_ocupado in otro_curso["bloques_ocupados"]:
                    if hay_tope(bloque_prueba, bloque_ocupado):
                        topes.append({
                            "RUT UC": rut,
                            "Nombre": curso_eval["Nombre"],
                            "Email": curso_eval["Email"],
                            "Sigla evaluacion": sigla_eval,
                            "NRC evaluacion": curso_eval["NRC"],
                            "Fecha evaluacion": bloque_prueba["fecha"],
                            "Dia evaluacion": bloque_prueba["dia_semana"],
                            "Inicio evaluacion": bloque_prueba["inicio_min"],
                            "Fin evaluacion": bloque_prueba["fin_min"],
                            "Sigla curso tope": otro_curso["Sigla"],
                            "NRC curso tope": otro_curso["NRC"],
                            "Bloque curso tope": f'{bloque_ocupado["dia_semana"]}:{bloque_ocupado["inicio_min"]}-{bloque_ocupado["fin_min"]}',
                            "Unidad Académica": curso_eval["Unidad Académica"],
                        })


In [106]:

df_topes = pd.DataFrame(topes).drop_duplicates().reset_index(drop=True)


In [107]:
df_topes

,RUT UC,Nombre,Email,Sigla evaluacion,NRC evaluacion,Fecha evaluacion,Dia evaluacion,Inicio evaluacion,Fin evaluacion,Sigla curso tope,NRC curso tope,Bloque curso tope,Unidad Académica
0,1162969,Paul Eisenblätter,eisenblaetter.paul@gmail.com,EAA3401,29585,2026-05-28,J,1050,1170,EAA3501,27961,J:1070-1160,Economia y Administracion


In [ ]:

df_topes.to_excel("topes_evaluaciones.xlsx", index=False)